In [ ]:
pip install pypdf2 requests pillow matplotlib

In [ ]:
import requests

# Download the 'Attention Is All You Need' paper
url = 'https://arxiv.org/pdf/1706.03762.pdf'
response = requests.get(url)
with open('attention_paper.pdf', 'wb') as f:
    f.write(response.content)
print('PDF downloaded successfully.')

In [ ]:
import PyPDF2
from PIL import Image
import io
import matplotlib.pyplot as plt

def extract_and_display_images(pdf_path):
    reader = PyPDF2.PdfReader(pdf_path)
    image_count = 0

    for page_index, page in enumerate(reader.pages):
        if '/Resources' in page and '/XObject' in page['/Resources']:
            xObject = page['/Resources']['/XObject'].get_object()

            for obj in xObject:
                if xObject[obj]['/Subtype'] == '/Image':
                    size = (xObject[obj]['/Width'], xObject[obj]['/Height'])
                    data = xObject[obj].get_data()

                    # Determine the image mode
                    if xObject[obj]['/ColorSpace'] == '/DeviceRGB':
                        mode = "RGB"
                    else:
                        mode = "P"

                    if '/Filter' in xObject[obj]:
                        if xObject[obj]['/Filter'] == '/FlateDecode':
                            img = Image.frombytes(mode, size, data)
                        elif xObject[obj]['/Filter'] == '/DCTDecode':
                            img = Image.open(io.BytesIO(data))
                        elif xObject[obj]['/Filter'] == '/JPXDecode':
                            img = Image.open(io.BytesIO(data))
                        else:
                            continue # Unsupported filter
                    else:
                        img = Image.frombytes(mode, size, data)

                    image_count += 1
                    plt.figure(figsize=(8, 8))
                    plt.imshow(img)
                    plt.title(f'Image {image_count} from Page {page_index + 1}')
                    plt.axis('off')
                    plt.show()

    if image_count == 0:
        print('No images found in the PDF.')

extract_and_display_images('attention_paper.pdf')

In [ ]:
import PyPDF2

def extract_text_with_context(pdf_path):
    reader = PyPDF2.PdfReader(pdf_path)

    print(f"--- Internal References and their Context Paragraphs ---\n")

    for page_num, page in enumerate(reader.pages):
        # Get the full text of the page to search for context
        page_text = page.extract_text()
        paragraphs = page_text.split('\n\n') # Rough split by double newlines

        if '/Annots' in page:
            annotations = page['/Annots']
            for annot in annotations:
                obj = annot.get_object()

                if obj.get('/Subtype') == '/Link':
                    # Check if it's an internal GoTo link
                    is_internal = False
                    if obj.get('/Dest') or (obj.get('/A') and obj.get('/A').get('/S') == '/GoTo'):
                        is_internal = True

                    if is_internal:
                        # Get the rectangle coordinates of the link to help find text if needed
                        # For this simple version, we'll look for keywords like 'Figure' or 'Table'
                        # in the vicinity or just show which page/link was found.
                        rect = obj.get('/Rect')

                        # Since standard PyPDF2 extract_text doesn't easily map Rect to text,
                        # we will display the paragraph from the page that likely contains a reference
                        # (heuristic: paragraphs containing 'Figure', 'Table', or numbers in brackets)

                        print(f"[Link detected on Page {page_num + 1}]")
                        # Finding the paragraph is tricky without coordinate-to-text mapping,
                        # so we'll display paragraphs from that page that look like references.
                        found_context = False
                        for para in paragraphs:
                            if any(word in para for word in ['Figure', 'Table', 'Tab.', 'Fig.', 'Section']):
                                print(f"Context: {para.strip()[:300]}...")
                                print("-" * 40)
                                found_context = True
                                break # Show one relevant paragraph per link detected

                        if not found_context:
                             print("Context: Internal reference detected (likely a citation or bibliography link).")
                             print("-" * 40)

extract_text_with_context('attention_paper.pdf')
